# 11.03 · The virtual filesystem and planning

**Module 11 · Deep Agents** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 1.5–2 h |
| 📋 Prerequisites | Lessons 11.01–11.02; Lesson 08.01 (checkpointers and threads). |
| 🔑 Keys | none for 🧪 cells · optional 💸 experiment ≈ 20–30K tokens (off by default) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Use** the file tools as the agent's working memory: seed files, let the agent write and edit, read them from the final state and render them.
2. **Explain** the default `StateBackend`: thread-scoped, persisted by your checkpointer, shared by the main agent and its subagents.
3. **Opt in** to planning with `TodoListMiddleware` and read `todos` from state.
4. **Restrict** an agent to read-only file tools.
5. **Test**, with repeated runs and an interval, whether planning helps your model on your task.

## 📚 Official reading (verified 2026-10-16, `deepagents` 0.7.26)

| Page | Read it for |
|---|---|
| [Deep Agents overview](https://docs.langchain.com/oss/python/deepagents/overview) | Virtual filesystem and task planning |
| [Backends: StateBackend](https://docs.langchain.com/oss/python/deepagents/backends) | Thread-scoped files, shared with subagents |
| [Built-in middleware: to-do list](https://docs.langchain.com/oss/python/langchain/middleware/built-in) | `TodoListMiddleware`, `write_todos` |
| [Multimodal inputs and outputs](https://docs.langchain.com/oss/python/deepagents/multimodal) | `read_file` on images and PDFs |
| [Changelog: v0.7.0](https://docs.langchain.com/oss/python/deepagents/changelog-py) | Why planning became opt-in; paginated `read_file`; `No files found` |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "deepagents>=0.7.23,<0.8" "langchain-google-genai>=4.4.1,<5" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("deepagents", "langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/11-deep-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GOOGLE_API_KEY', 'GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm11-filesystem-planning'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.deep import deep_key_ready, deep_model_spec, fallback_specs

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")

DEEP_MODEL = deep_model_spec("main")              # MASTERY_DEEP_MODEL (course default: Gemini free tier)
SUBAGENT_MODEL = deep_model_spec("subagent")      # MASTERY_DEEP_SUBAGENT_MODEL
HAVE_DEEP_KEY = deep_key_ready(DEEP_MODEL)        # the key for THAT provider (GOOGLE_API_KEY for google_genai:)
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m11-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("deep model:", DEEP_MODEL, "| subagents:", SUBAGENT_MODEL)
print("fallbacks :", fallback_specs(DEEP_MODEL) or "none (optional: MASTERY_FALLBACK_MODELS in .env)")
if not HAVE_DEEP_KEY:
    print(f"\n⚠️ No key for {DEEP_MODEL.split(':')[0]}: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Helpers for scripted runs: a tool call, a user turn, the tool results of a run.
from langchain_core.messages import AIMessage

from mastery.deep import build_deep_agent, run_stats
from mastery.testing import ScriptedChatModel


def call(name: str, args: dict, call_id: str = "1") -> AIMessage:
    return AIMessage("", tool_calls=[{"name": name, "args": args, "id": call_id}])


def user(text: str) -> dict:
    return {"messages": [{"role": "user", "content": text}]}


def tool_results(result) -> list[str]:
    return [m.content for m in result["messages"] if m.type == "tool"]

---
## Part 1 · Files as working memory

Long tasks produce intermediate artifacts: notes, drafts, raw search results. Keeping them in **files** instead of the chat means the model reads only what it needs, when it needs it (context offloading). Files are in the result's `files` key; you seed them the same way.

In [ ]:
# 🧪 Seed files into state, let the agent read and write, read the files back from the FINAL state.
from deepagents.backends.utils import create_file_data

script = ScriptedChatModel(responses=[
    call("read_file", {"file_path": "/brief.md"}),
    call("write_file", {"file_path": "/draft.md", "content": "# Welcome\n- Read the handbook\n- Meet your buddy"}, "2"),
    call("edit_file", {"file_path": "/draft.md", "old_string": "Meet your buddy", "new_string": "Meet your buddy on day 1"}, "3"),
    AIMessage("Draft ready in /draft.md")])
agent = build_deep_agent(script, fallbacks=False)
seeded = agent.invoke({**user("Draft a welcome note from the brief."),
                       "files": {"/brief.md": create_file_data("Audience: new hires. Two bullets.")}})
for text in tool_results(seeded):
    print("tool →", text.replace("\n", " ⏎ ")[:90])
print("\nfiles in state:", list(seeded["files"]))

**You should now see** the brief read back (with a `@@ lines 1-1 of 1 @@` header: `read_file` is paginated and says what is left), `Updated file /draft.md`, `Successfully replaced 1 instance(s)…`, and `files in state: ['/brief.md', '/draft.md']`. `edit_file` replaces an exact string, like the one you wrote in Lesson 11.01.

In [ ]:
# 🧪 Render an agent-written file in the notebook (Jupyter skill: files are artifacts you can show).
from IPython.display import Markdown, display

display(Markdown(seeded["files"]["/draft.md"]["content"]))

## Part 2 · Where files live: `StateBackend`

In [ ]:
# 🧪 StateBackend is THREAD-scoped: files survive turns of one thread (checkpointer), not other threads.
from langgraph.checkpoint.memory import InMemorySaver

script = ScriptedChatModel(responses=[
    call("write_file", {"file_path": "/plan.md", "content": "step 1"}), AIMessage("written"),
    call("ls", {"path": "/"}, "2"), AIMessage("listed"),
    call("ls", {"path": "/"}, "3"), AIMessage("listed")])
agent = build_deep_agent(script, checkpointer=InMemorySaver(), fallbacks=False)
agent.invoke(user("write a plan"), {"configurable": {"thread_id": "t1"}})
same = agent.invoke(user("what files do we have?"), {"configurable": {"thread_id": "t1"}})
other = agent.invoke(user("what files do we have?"), {"configurable": {"thread_id": "t2"}})
print("thread t1, turn 2:", tool_results(same)[-1])
print("thread t2        :", tool_results(other)[-1])

**What just happened?** With the default `StateBackend`, files are part of the graph state of **one thread**: the checkpointer brings them back on the next turn of `t1`, and `t2` starts empty (`No files found`). Without a checkpointer they last one `invoke`. Lesson 11.04 adds a store for files that must outlive a thread.

In [ ]:
# 🧪 Subagents share the thread's files: the general-purpose subagent writes, the main agent reads.
script = ScriptedChatModel(responses=[      # one model: main and subagent take turns
    call("task", {"subagent_type": "general-purpose", "description": "Write 3 facts to /facts.md"}),
    call("write_file", {"file_path": "/facts.md", "content": "- a\n- b\n- c"}, "s1"),   # subagent
    AIMessage("Wrote /facts.md"),                                                         # subagent's answer
    call("read_file", {"file_path": "/facts.md"}, "2"),                                   # main agent
    AIMessage("The subagent's facts are in /facts.md")])
shared = build_deep_agent(script, fallbacks=False).invoke(user("Get 3 facts via a subagent, then check them."))
print("main agent read:", tool_results(shared)[-1].replace("\n", " ⏎ "))

**What just happened?** The subagent wrote `/facts.md`, and the main agent read it afterwards: the docs say the `StateBackend` is shared by the supervisor and its subagents, and files a subagent writes stay after it finishes. That is how subagents hand over large results without pasting them into the chat (Lesson 11.05).

> ⚠️ **Production note: shared files are shared risk.** Any subagent can overwrite any file. In production: give subagents their own paths (`/research/web/`, `/research/docs/`), and restrict writes with permissions (Lesson 11.04).

## Part 3 · Opt-in planning

In [ ]:
# 🧪 Opt-in planning: TodoListMiddleware adds write_todos and a `todos` channel in state.
from langchain.agents.middleware import TodoListMiddleware

script = ScriptedChatModel(responses=[
    call("write_todos", {"todos": [{"content": "Read the brief", "status": "in_progress"},
                                   {"content": "Write the draft", "status": "pending"}]}),
    call("write_todos", {"todos": [{"content": "Read the brief", "status": "completed"},
                                   {"content": "Write the draft", "status": "completed"}]}, "2"),
    AIMessage("Both steps done")])
planned = build_deep_agent(script, middleware=[TodoListMiddleware()], fallbacks=False).invoke(user("Plan, then work."))
for todo in planned["todos"]:
    print(f"{todo['status']:<12} {todo['content']}")
print("tools:", script.bound_tools[-1], "| todos are in state, like files")

**You should now see** two todos moving from `in_progress` / `pending` to `completed`, and `write_todos` added to the tools. This is the production version of your Lesson 11.01 todo tool, with three statuses. v0.7 made it **opt-in**: the official evals showed little gain on basic tasks, and the schema and instructions cost tokens on every call. Whether it helps **your** model on **your** long tasks is an empirical question: Part 5.

## Part 4 · Read-only agents and multimodal reads

In [ ]:
# 🧪 A read-only agent: pass your own FilesystemMiddleware with a tool allow-list (it replaces the default).
from deepagents import FilesystemMiddleware

script = ScriptedChatModel(responses=[AIMessage("I can only look.")])
build_deep_agent(script, middleware=[FilesystemMiddleware(tools=["ls", "read_file", "glob", "grep"])],
                 fallbacks=False).invoke(user("hi"))
print("tools:", script.bound_tools)

**What just happened?** A middleware whose name matches a built-in **replaces** it (v0.7), so your `FilesystemMiddleware(tools=[...])` is the only one: no `write_file`, `edit_file` or `delete`. A reviewer or auditor subagent should look like this.

**Multimodal reads:** `read_file` returns images (`.png`, `.jpg`, `.gif`, `.webp`) as image content blocks, and since v0.5 also PDFs, audio and video, if your model accepts them (Gemini does). Seed a binary file with `create_file_data(base64_text, encoding="base64")`. Try it in Exercise 4.

## Part 5 · 💸 Does planning help? Measure it

> 💸 Calls your deep model (`MASTERY_DEEP_MODEL`, default Gemini free tier). Rate-limited to one call every 4 s and capped per run. Without the key it prints `⏭️ Skipped`.

In [ ]:
# 💸 LIVE (opt-in): does planning help YOUR model on this task? REPEATS × 2 runs, ≈ 3–5K tokens per run.
RUN_EXPERIMENT = False          # True: runs 2 × REPEATS deep-agent runs
REPEATS = 3
TASK = ("Create /report.md with three sections: '## Goal', '## Steps' (exactly 3 numbered steps) and '## Risks' "
        "(2 bullets), about onboarding a new teammate. Then reply DONE.")


def report_ok(result) -> bool:
    data = (result.get("files") or {}).get("/report.md")
    text = data["content"] if isinstance(data, dict) else ""
    steps = [ln for ln in text.splitlines() if ln.strip()[:2] in ("1.", "2.", "3.")]
    return all(h in text for h in ("## Goal", "## Steps", "## Risks")) and len(steps) == 3


if RUN_EXPERIMENT and HAVE_DEEP_KEY:
    from mastery.evals.stats import wilson_interval

    rows = []
    for label, extra in [("no planning", []), ("TodoListMiddleware", [TodoListMiddleware()])]:
        wins, tokens = 0, 0
        for _ in range(REPEATS):
            out = build_deep_agent(middleware=extra, model_calls=10, tool_calls=10).invoke(user(TASK))
            wins += report_ok(out)
            stats = run_stats(out)
            tokens += stats["input_tokens"] + stats["output_tokens"]
        rows.append({"variant": label, "pass": f"{wins}/{REPEATS}",
                     "95% CI": str(wilson_interval(wins, REPEATS)).split(" (")[0], "tokens/run": tokens // REPEATS})
    table(rows, ["variant", "pass", "95% CI", "tokens/run"])
elif not RUN_EXPERIMENT:
    print("⏭️ Skipped: set RUN_EXPERIMENT = True to run it (≈ 20–30K tokens with REPEATS = 3).")
else:
    print(f"⏭️ Skipped: no key for {DEEP_MODEL} (see the setup cell).")

**You should now see** (when enabled) two rows with a pass rate, a 95% interval and tokens per run. With 3 runs each, the intervals almost always overlap: you can't claim a difference. Report it honestly as "no evidence either way at n = 3", and decide on cost: if planning doesn't measurably help, leave it off. Lesson 11.11 gives you a proper dataset.

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| Files gone on the next turn | No checkpointer, or a different `thread_id` | Pass `checkpointer=` and reuse the `thread_id` |
| `KeyError: 'todos'` | `TodoListMiddleware` not added | `middleware=[TodoListMiddleware()]` |
| Your parser of `read_file` output breaks | v0.7 dropped the `cat -n` gutter and added a `@@ lines … @@` header | Read `result["files"]` instead of parsing tool text |
| `ls` returns `No files found`, not `[]` | v0.7 output format | Compare with the string, or use `result["files"]` |
| Seeded file ignored | Plain string instead of `create_file_data(...)` | Seed `{path: create_file_data(text)}` |

## 🏋️ Exercises

1. **(Easy)** Seed two files and script a `glob` call for `/*.md`. What does it return?
2. **(Medium)** Run the Part 2 agent on thread `t1` a third time with an `edit_file` call. Is the edit there on turn 4?
3. **(Medium)** Write `todo_progress(state) -> str` that renders `"2/3 completed"` from `state["todos"]`, and show it after the Part 3 run.
4. **(Stretch, 💸)** Seed a small PNG (base64) and ask your deep model to describe it with `read_file`. Which content block type reaches the model?

<details>
<summary><b>Show solutions</b></summary>

**1.** The matching paths, one per line (or `No files found`).
```python
s = ScriptedChatModel(responses=[call("glob", {"pattern": "/*.md"}), AIMessage("ok")])
out = build_deep_agent(s, fallbacks=False).invoke({**user("find"), "files": {
    "/a.md": create_file_data("a"), "/b.md": create_file_data("b")}})
print(tool_results(out))
```

**2.** Yes: every turn of `t1` starts from the last checkpoint, files included. Add `call("edit_file", ...)` and a final `AIMessage` to a new script, rebuild the agent with the **same** checkpointer object, and invoke on `t1`.

**3.**
```python
def todo_progress(state) -> str:
    todos = state.get("todos") or []
    return f"{sum(t['status'] == 'completed' for t in todos)}/{len(todos)} completed"
print(todo_progress(planned))
```

**4.** An image content block (`type: image`, base64 data). Keep images small: they count as input tokens.
```python
import base64
png = base64.b64encode((REPO_ROOT / "path" / "to" / "your.png").read_bytes()).decode()
out = build_deep_agent(model_calls=4).invoke({**user("Describe /pic.png in one sentence."),
                                             "files": {"/pic.png": create_file_data(png, encoding="base64")}})
```
</details>

## 🔎 Docs drill (5 min)

With the default `StateBackend`, can a subagent read a file the main agent wrote earlier in the same run, and does a file a subagent writes survive after the subagent finishes?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Backends](https://docs.langchain.com/oss/python/deepagents/backends) → StateBackend.
2. Yes to both: the backend is shared between the supervisor and subagents, and files a subagent writes remain in state after it completes, available to the supervisor and other subagents.
3. You verified the second half in Part 2 (`/facts.md`).

</details>

## ✅ Checkpoint

- [ ] I can seed files, read them from the final state and render them.
- [ ] I can explain why files survive turns of one thread but not other threads.
- [ ] I can add planning, read `todos`, and say what it costs per call.
- [ ] I can build a read-only agent.
- [ ] Restart & Run All passes without keys; outputs cleared before saving.

## 🔎 Research checkpoint

In the [changelog](https://docs.langchain.com/oss/python/deepagents/changelog-py) and the [to-do list middleware docs](https://docs.langchain.com/oss/python/langchain/middleware/built-in), find why planning became opt-in in v0.7 and whether any harness profile still turns it on by default (2026-10-16: the OpenAI Codex profile does). Log your answer and your Part 5 result (or why you skipped it) in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Files are the agent's working memory: seeded with `create_file_data`, read back from `result["files"]`.
- `StateBackend` is thread-scoped (checkpointer) and shared with subagents.
- Planning (`TodoListMiddleware`) is opt-in: measure it before paying for it on every call.
- A `FilesystemMiddleware(tools=[...])` allow-list makes a read-only agent.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 11-04-backends-routing-permissions.ipynb** → [`11-04-backends-routing-permissions.ipynb`](11-04-backends-routing-permissions.ipynb).

[Module 11 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)